## PX1 Mini-Cam — определение файловой системы SD-образа

Вторая проверка после отчёта PX1_SD_REPORT.txt: раздел Linux-типа в MBR начинается с 32 901 120-го байта, поэтому прежние 16 МиБ его не захватывали.

Этот блокнот **только читает** первые 34 МиБ распакованного потока с Google Диска (сам файл 2,93 ГБ не загружается на Android). Он определяет тип файловой системы и сохраняет короткий `PX1_SD_FS_REPORT.txt` в корне того же Google Диска. Не читает персональные файлы, не выгружает образ и не проверяет сторонние серверы.

В Chrome на Android открой блокнот → **Среда выполнения → Выполнить все** → предоставь Google Colab запрошенный доступ к Google Диску. После завершения напиши ChatGPT «Отчёт по разделу готов».


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, struct, os
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
source = root / 'Untitled folder' / 'ЛП.imgc'
if not source.is_file():
    source = next((Path(d) / 'ЛП.imgc' for d, _, fs in os.walk(root) if 'ЛП.imgc' in fs), None)
if source is None or not source.is_file():
    raise FileNotFoundError('ЛП.imgc не найден на подключённом Google Диске')
toolpath = Path('/content/unimgc')
if not toolpath.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Shizmob/unimgc.git', str(toolpath)], check=True)
subprocess.run(['make', '-C', str(toolpath)], check=True)
p = subprocess.Popen([str(toolpath/'unimgc'), str(source)], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
try:
    data = p.stdout.read(34 * 1024 * 1024)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
if len(data) < 512 or data[510:512] != b'\x55\xaa':
    raise ValueError('Не найден MBR. Нужна ручная проверка содержимого образа.')
lines = ['PX1 Mini-Cam: проверка файловой системы (без изменения образа)', 'Архив: ЛП.imgc']
for i in range(4):
    entry = data[446 + 16*i : 446 + 16*(i+1)]
    typecode = entry[4]
    lba, count = struct.unpack_from('<II', entry, 8)
    if not (typecode and count): continue
    offset = lba * 512
    lines.append(f'Раздел {i+1}: тип=0x{typecode:02X}; смещение={offset}; размер={count*512} байт')
    if offset + 2048 > len(data):
        lines.append('  Раздел начинается за границей прочитанного фрагмента')
        continue
    boot = data[offset:offset+4096]
    sb = boot[1024:2048]
    if sb[56:58] == b'\x53\xef':
        block_size = 1024 << struct.unpack_from('<I', sb, 24)[0]
        inode_count = struct.unpack_from('<I', sb, 0)[0]
        volume_label = sb[120:136].split(b'\0')[0].decode('utf-8', errors='replace')
        lines.append('  Файловая система: семейство ext2/ext3/ext4 (проверена сигнатура)')
        lines.append(f'  Размер блока: {block_size}; inode: {inode_count}; метка: {volume_label or "(пустая)"}')
    elif boot[3:11] == b'NTFS    ': lines.append('  Файловая система: NTFS')
    elif boot[3:11] == b'EXFAT   ': lines.append('  Файловая система: exFAT')
    elif b'FAT32' in boot[64:95]: lines.append('  Файловая система: FAT32')
    elif b'FAT16' in boot[32:95]: lines.append('  Файловая система: FAT16')
    else: lines.append('  Файловая система: пока неизвестна')
lines.append('Этот отчёт не подтверждает наличие ОС или уязвимостей.')
report = root/'PX1_SD_FS_REPORT.txt'
report.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines))
print('Отчёт записан:', report.name)